In [3]:
%pip install shaped pandas
import pandas as pd


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import urllib.request
import tarfile
import os

# Tar file URL
tar_url = "http://mtg.upf.edu/static/datasets/last.fm/lastfm-dataset-1K.tar.gz" 
raw_data_dir = "data/raw" 

os.makedirs(raw_data_dir, exist_ok=True)

print(f"Downloading {tar_url}...")
tar_path = os.path.join(raw_data_dir, "downloaded_file.tar.gz")
urllib.request.urlretrieve(tar_url, tar_path)
print(f"Downloaded to {tar_path}")

print(f"Extracting to {raw_data_dir}...")
with tarfile.open(tar_path, 'r:gz') as tar:  # Use 'r:' for uncompressed tar, 'r:gz' for .tar.gz
    tar.extractall(path=raw_data_dir)

print("Extraction complete!")

os.remove(tar_path)
print("Cleaned up tar file")

KeyboardInterrupt: 

In [6]:
df = pd.read_csv('data/raw/userid-profile.tsv', sep='\t')
df = df.rename(columns={'#id': 'user_id'})
df.columns
df['registered'] = pd.to_datetime(df['registered'], format='%b %d, %Y')
df.to_json('data/processed/lastfm_users.jsonl', orient='records', lines=True)
print('Users table created')

Users table created


Now we read the interactions table. The table has 14M records; we only keep the most recent 500 items for each user. This gives us some sequential information about tracks. 

In [4]:
df = pd.read_csv('data/raw/userid-timestamp-artid-artname-traid-traname.tsv', 
    names=['user_id', 'created_at', 'artist_id', 'artist_name', 'item_id', 'track_name'],
    sep='\t', 
    on_bad_lines='skip')
df.columns
df.shape
most_recent_interactions = df.sort_values('created_at', ascending=False).groupby('user_id').head(500)
most_recent_interactions.to_json('data/processed/lastfm_interactions.jsonl', orient='records', lines=True)
print(f'Interactions table created (500 interactions per user only) - {most_recent_interactions.shape[0]} rows')

Interactions table created (500 interactions per user only) - 475506 rows


In [15]:
tracks = most_recent_interactions.drop_duplicates(subset=['item_id'], keep='first')
tracks = tracks.drop(columns=['user_id', 'timestamp'])
tracks.to_json('data/processed/lastfm_items.jsonl', orient='records', lines=True)
print(f'Items table created - {tracks.shape[0]} tracks')

Items table created - 159975 tracks


## Upload datasets to Shaped

```bash
shaped create-dataset-from-uri --name lastfm_interactions --path data/processed/lastfm_interactions.jsonl --type jsonl
shaped create-dataset-from-uri --name lastfm_users --path data/processed/lastfm_users.jsonl --type jsonl
shaped create-dataset-from-uri --name lastfm_items --path data/processed/lastfm_items.jsonl --type jsonl

```